# Isochrone Catchment Pipeline - Bayern

1. Fix params, change `SPEC` accordingly
2. Data loading: `panel`, admin. shapefile, and pop. raster
3. Optional: Restrict to a Bundesland; consult `config.py` for a list of Schlussel indicating Bundeslaende. 
4. Snap stores (given specification) to the road network (server must be running at the given `OSRM` address). Results cached; same Shop Type, year specification and road network can be re-run by pointing to the resulting cache file. Not terribly expensive.
5. Reduce, addapt and snap pop cells: If the specification-corresponding parquet does not exist, load pop raster, addapt resolution (lower resolution where catchments are wide, given population, store locations), and snap to the road network. This is computationally taxing; the pop. raster is very fine definition, and millions of cells need to snap to the network. They cannot be passed in bulk, like the distance-to querries, and therefore we querry the server millions of times. This is why the local implementation is of most value, because this would take hours if not days if we had to wait a sec after every querry. Snapping does not move the cell, but masks it with the road-approximate cell; then it inherits its attributes when the nearest store is found. After snapping, duplicate masks (cell masks that coincide) are dropped (aggregated, and the corresponding cells inherit the survivors' attributes)
6. Test server for the big task 
7. Travel times: 


Order matters in two places:

1. Stores are snapped before `adapt_resolution`, because resolution is chosen
   from the distance to the nearest store and that should be the routable position.
2. `m1` / `m2` are saved the moment they exist. They are produced by
   `group_by` + `with_row_index`, and Polars does not guarantee group order across
   runs -- regenerating them later yields *different* node ids than the ones baked
   into `cand` / `dur` / the assignments, and the mismatch fails silently.

Only `snap` and `travel_times` touch OSRM; both cache, so a re-run is cheap.

**Bump `SPEC` whenever you change a reduction or candidate parameter.** 
`c2`, `mapping` and `matrix.npz` are one artefact split across three files; a stale
matrix beside regenerated cells is the main way this pipeline produces wrong
answers, and only some of those combinations raise on their own.

## 0. Config

In [1]:
import sys
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))
from scripts.tdlinx.data_loading import load_geodata

import scripts.isochrones.network_voronoi_2 as nv
import scripts.isochrones.market_metrics as M

# ---- paths -----------------------------------------------------------------
SHARE   = Path(r"X:\Project-Shares\germart-lite\germart-papaio")
PANEL   = SHARE / "panel"
GHSL_NAT = r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project_2\data\ghsl_raster_files\ghs_pop_germany_clipped.tif"
WORK    = SHARE / r"edeka_project_2\data\catchments"
OUT     = SHARE / r"edeka_project_2\outputs\catchments"
OSRM        = "http://localhost:5000"      # or http://192.168.70.123:5000

# ---- parameters ------------------------------------------------------------
YEARS       = [2024]                       # add more; only assign() re-runs per year
SHOP_TYPES  = ["Supermarkt", "Discount Geschäft"]
K           = 20                           # starting k; refine_until widens it
CERT_TARGET = 0.90                         # widen k until this share is certified
CERT_RANK   = 1                            # rank the certification target applies to
RANKS       = 1                            # WTA headline: rank 1 is all that is used
ISO_MIN     = 20                           # local-market isochrone, minutes
ATTRACT     = "VKFL"                       # carried through; sales-area shares
LAND        = "Baden-Württemberg"
KREIS_ID    = "AGS5"                       # matches MARKET_DEFINING_COLS and the panel
KREIS_NAME  = "GEN"

# SPEC namespaces every derived artefact. Change any reduction or candidate
# parameter above and you MUST bump this: cells, mapping and matrix are one object
# split across three files, and mixing versions is the main way this goes wrong.
#------------------------#
SPEC    = "bw_supermarket_2024"
#------------------------#

CACHE   = OUT / SPEC / "cache"
CKPT    = OUT / SPEC / "ckpt"
SPECDIR = WORK / SPEC; OUT = OUT / SPEC
FIGS    = OUT / "figures"
for d in (WORK, CACHE, SPECDIR, CKPT, OUT, FIGS): d.mkdir(parents=True, exist_ok=True)

# GHSL is rebound in 1b to the Bayern clip. Keeping the national path under its own
# name means re-running this cell can no longer leave GHSL pointing at a different
# grid from the cells/mapping already loaded -- which matches on cell_id through an
# inner join and produces wrong answers rather than an error.
GHSL = GHSL_NAT


## 1. Load panel + geodata

In [2]:
panel = pl.read_parquet(PANEL)


In [3]:
panel.group_by(pl.col("Shop Type")).len()

Shop Type,len
str,u32
"""Drogeriemarkt""",59280
"""Supermarkt""",197098
"""Discount Geschäft""",190925
"""Baumarkt""",2335
"""Getränkemarkt""",121364


In [4]:
panel = pl.read_parquet(PANEL)
gdf, kreis_gdf, bundesland_gdf = load_geodata()

panel = panel.filter(pl.col("Shop Type").is_in(SHOP_TYPES), pl.col("year").is_in(YEARS))
print(f"panel: {panel.height:,} rows | {panel['TDLinx'].n_unique():,} stores | "
      f"years {panel['year'].min()}-{panel['year'].max()}")

panel: 31,213 rows | 31,213 stores | years 2024-2024


## 1b. Restrict to bw

Clips the inputs: a bw-only raster, bw Kreise, and a panel
filtered to bw stores. Everything downstream runs unchanged.

`GHSL` and `kreis_gdf` are rebound here, so `load_cells`, `rasterize_regions` and
`write_label_raster` all see the clipped grid and stay mutually consistent.

In [5]:
import geopandas as gpd, rasterio
from rasterio.mask import mask as rio_mask

AGS5_PREFIX = "08"                        # bw; see SCHLUESSEL_TO_ABR in config

# ---- Kreis GEOMETRY: dissolve the Gemeinde layer, filtered first ------------
# Filtering before the dissolve: no reason to dissolve ~11k German Gemeinden to
# then throw 85% of them away.
gem = gdf.assign(AGS5=gdf["ARS"].astype(str).str[:5])
gem = gem[gem["AGS5"].str.startswith(AGS5_PREFIX)]
assert len(gem), f"no ARS starting {AGS5_PREFIX!r}"
kreis_geom = gem.dissolve(by="AGS5")[["geometry"]].reset_index()

# ---- Kreis NAMES: from the Kreis layer, never from the dissolve -------------
# dissolve(aggfunc="first") kept whichever Gemeinde happened to sort first, so
# "GEN" was an arbitrary municipality name, not the Kreis. Gemeinde names are also
# not unique across Kreise, so filtering figures on it could silently match two.
kr = kreis_gdf.copy()                     # the layer load_geodata() already returns
_key = next((c for c in ("AGS", "ARS", "RS", "AGS5", "SCH", "ARS_0", "AGS_0")
             if c in kr.columns), None)
if _key is not None and KREIS_NAME in kr.columns:
    kr["AGS5"] = kr[_key].astype(str).str.zfill(5).str[:5]
    names = kr.loc[kr["AGS5"].str.startswith(AGS5_PREFIX),
                   ["AGS5", KREIS_NAME]].drop_duplicates("AGS5")
    kreis_gdf = kreis_geom.merge(names, on="AGS5", how="left")
else:
    print(f"WARNING kreis layer has no usable key/{KREIS_NAME}; falling back to AGS5")
    kreis_gdf = kreis_geom.assign(**{KREIS_NAME: kreis_geom["AGS5"]})

_miss = kreis_gdf[KREIS_NAME].isna()
if _miss.any():
    print(f"WARNING {_miss.sum()} Kreis(e) without a name; using AGS5 for those")
    kreis_gdf.loc[_miss, KREIS_NAME] = kreis_gdf.loc[_miss, "AGS5"]

# München exists twice in Bayern (kreisfreie Stadt 09162 and Landkreis 09184), and
# so do several others, so the *name* is not a key even when it is correct. Build
# an unambiguous label for titles and filenames; keep AGS5 as the only key.
_dup = kreis_gdf[KREIS_NAME].duplicated(keep=False)
kreis_gdf["KREIS_LABEL"] = kreis_gdf[KREIS_NAME].astype(str)
kreis_gdf.loc[_dup, "KREIS_LABEL"] = (kreis_gdf.loc[_dup, KREIS_NAME].astype(str)
                                      + " (" + kreis_gdf.loc[_dup, "AGS5"] + ")")
assert kreis_gdf["AGS5"].is_unique and kreis_gdf["KREIS_LABEL"].is_unique

bw = kreis_gdf.dissolve().geometry.iloc[0]

# ---- raster -> Bayern extent ----------------------------------------------
GHSL_BY = SPECDIR / f"ghs_pop_{LAND.lower()}_100m.tif"
if not GHSL_BY.exists():
    with rasterio.open(GHSL_NAT) as src:
        geom = gpd.GeoSeries([bw], crs=kreis_gdf.crs).to_crs(src.crs).geometry.values
        img, tf = rio_mask(src, geom, crop=True, filled=True, nodata=0)
        prof = src.profile
    prof.update(height=img.shape[1], width=img.shape[2], transform=tf,
                nodata=0, compress="lzw")
    with rasterio.open(GHSL_BY, "w", **prof) as dst:
        dst.write(img[0], 1)
    print(f"wrote {GHSL_BY.name}  {prof['height']}x{prof['width']}")

GHSL = GHSL_BY                            # everything downstream uses this grid

# ---- panel -> Bayern -------------------------------------------------------
# NOTE (open): this drops stores just over the Land border, so every border Kreis
# sees fewer competitors than it has -- catchment_pop inflated, HHI biased up.
# Fine while Bayern is a speed proxy for the national run; must not survive into
# a published Bayern-only number.
panel = panel.filter(pl.col("Bundesland") == "Nordrhein-Westfalen")

print(f"panel : {panel.height:,} rows | {panel['TDLinx'].n_unique():,} stores")
print(f"Kreise: {len(kreis_gdf)}")
print(kreis_gdf[["AGS5", KREIS_NAME]].head())


wrote ghs_pop_baden-württemberg_100m.tif  2390x2234
panel : 5,673 rows | 5,673 stores
Kreise: 44
    AGS5          GEN
0  08111    Stuttgart
1  08115    Böblingen
2  08116   Denkendorf
3  08117  Böhmenkirch
4  08118       Asperg


## 2. Stores -> snapped

`drop_over_m=300`: a real operating retail location has a road within 300 m. Anything further is a geocode to check, and because a store is a candidate for thousands of cells one bad row corrupts a whole region.

In [6]:
stores = nv.prepare_stores(panel, keep=("Gruppe", "Shop Type"))
stores = nv.snap(stores, base_url=OSRM, cache_path=CACHE, drop_over_m=300)
stores.write_parquet(SPECDIR / f"stores_snapped_{SPEC}.parquet")

prepare_stores: 5,673 panel rows -> 5,673 unique stores
  snap: 32/256 chunks
  snap: 64/256 chunks
  snap: 96/256 chunks
  snap: 128/256 chunks
  snap: 160/256 chunks
  snap: 192/256 chunks
  snap: 224/256 chunks
  snap: 256/256 chunks
snap: 5,673 point(s) requested in 22s
  snap distance m: p50=11 p95=41 p99.9=137 max=376
  dropped 3 point(s) snapping >300m


In [7]:
# stores = pl.read_parquet(SPECDIR / f"stores_snapped_{SPEC}.parquet")

## 3. Cells -> reduced + snapped

Run once. Every later run reads the parquet and never touches OSRM.

In [8]:

cells_path, map_path = SPECDIR / f"cells_solve_{SPEC}.parquet", SPECDIR / f"mapping_{SPEC}.parquet"

if cells_path.exists() and map_path.exists():
    cells_orig = nv.load_cells(GHSL)                 # local, deterministic, no OSRM
    c2      = pl.read_parquet(cells_path)
    mapping = pl.read_parquet(map_path)
    print(f"loaded {c2.height:,} solve nodes from disk")
else:
    cells_orig = nv.load_cells(GHSL)
    c1, m1 = nv.adapt_resolution(cells_orig, stores)
    c1     = nv.snap(c1, base_url=OSRM, cache_path=CACHE)
    c2, m2 = nv.dedupe(c1, tol_m=0.0)                # 0 = exact, no approximation
    mapping = nv.compose(m1, m2)

    # save immediately: group_by order is not stable across runs
    m1.write_parquet(SPECDIR / f"map_adapt_{SPEC}.parquet")
    m2.write_parquet(SPECDIR / f"map_dedupe_{SPEC}.parquet")
    mapping.write_parquet(map_path)
    c2.write_parquet(cells_path)

print(f"{cells_orig.height:,} raster cells -> {c2.height:,} solve nodes "
      f"({c2.height/cells_orig.height:.1%})")

load_cells: 773,191 populated cells, 10,999,308 people
adapt_resolution: 773,191 -> 42,873 cells (5.5%); population delta 0.0e+00
  snap: 32/256 chunks
  snap: 64/256 chunks
  snap: 96/256 chunks
  snap: 128/256 chunks
  snap: 160/256 chunks
  snap: 192/256 chunks
  snap: 224/256 chunks
  snap: 256/256 chunks
snap: 42,873 point(s) requested in 153s
  snap distance m: p50=34 p95=495 p99.9=1805 max=3120
  WARNING 1 point(s) failed to snap at all
dedupe(tol=0m): 42,873 -> 42,504 nodes (99.1%)
773,191 raster cells -> 42,504 solve nodes (5.5%)


## 4. Preflight

10000 is a good value for `max_table_size` in the server. If observed speed exceeds `V_MAX_KMH`, the certification bound is unsound -- raise it in the module.

In [9]:
pf = nv.preflight(c2, base_url=OSRM)
MAX_COORDS = max(1000, pf["max_coords"])   # fed straight into travel_times below

preflight: /table accepts a 2500 x 2500 matrix (5,000 coords). Pass max_coords=5000 to travel_times.
preflight: fastest route-average 100 km/h (a LOWER bound on link speed) vs V_MAX_KMH=180.


In [10]:
pf

{'max_coords': 5000,
 'max_table_size_at_least': 5000,
 'mean_route_speed_kmh_max': 100.49174332039277}

In [11]:
MAX_COORDS

5000

## 5. Travel times

Checkpointed: kill and re-run and it resumes. `progress_every` prints a measured ETA after the first batches.

In [12]:
matrix_path = SPECDIR / f"matrix_{SPEC}_temp.npz"

if matrix_path.exists():
    z    = nv.load_matrix(matrix_path, stores=stores, cells=c2)
    cand, dur = z["candidates"], z["durations"]
    print(f"loaded matrix {dur.shape}")
else:
    cand   = nv.candidates(c2, stores, k=K)
    dur, _ = nv.travel_times(c2, stores, cand, base_url=OSRM, max_coords=MAX_COORDS,
                             checkpoint=CKPT / f"{SPEC}_temp", max_workers=8, progress_every=50)
    nv.save_matrix(matrix_path, c2, stores, cand, dur)

nv.certification_report(cand, dur, ranks=min(3, cand.k))


travel_times: 2/2 blocks to solve, ~1M matrix entries
travel_times: done in 3.4 min
  rank 1: 0.00% certified
  rank 2: 0.00% certified
  rank 3: 0.00% certified


Certification is not a formality: an uncertified node is one where a store outside
the candidate set *could* be nearer, so the catchment it was given is unproven.

`refine_until` escalates k until `CERT_TARGET` of nodes are certified at
`CERT_RANK`, re-solving only the nodes still uncertified at each step, and each
pass gets its own checkpoint directory (the candidate set changes every pass, so
sharing one would trip the fingerprint check).

It rebinds `cand` and `dur` **in place** and saves immediately. That pairing is
the point: rebinding only some of `cand` / `dur` / `matrix.npz` is what leaves the
numbers in this notebook disagreeing with the matrix on disk, so that the next run
of the same `SPEC` quietly produces different results.

In [13]:
cand, dur, cert = nv.refine_until(
    c2, stores, cand, dur,
    target=CERT_TARGET, rank=CERT_RANK, ks=(40, 60, 90, 140),
    base_url=OSRM, max_coords=MAX_COORDS, checkpoint=CKPT / "refine_temp",
)
nv.save_matrix(matrix_path, c2, stores, cand, dur, cert=cert)
nv.certification_report(cand, dur, ranks=min(3, cand.k))


refine_until: start k=20, rank-1 certified 0.00%, target 90%
refine: 42,504/42,504 uncertified at k=20, retrying at k=40
travel_times: 2/2 blocks to solve, ~2M matrix entries
travel_times: done in 3.0 min
refine: certified 0.00% -> 0.00%
refine: 42,504/42,504 uncertified at k=40, retrying at k=60
travel_times: 2/2 blocks to solve, ~3M matrix entries
travel_times: done in 3.1 min
refine: certified 0.00% -> 0.00%
refine: 42,504/42,504 uncertified at k=60, retrying at k=90
travel_times: 2/2 blocks to solve, ~4M matrix entries
travel_times: done in 3.2 min
refine: certified 0.00% -> 0.00%
refine: 42,504/42,504 uncertified at k=90, retrying at k=140
travel_times: 2/2 blocks to solve, ~6M matrix entries
travel_times: done in 3.7 min
refine: certified 0.00% -> 0.00%
WARNING refine_until: stopped at k=140 with 0.00% certified, below the 90% target. Extend `ks`, or accept and report the rate -- uncertified does not mean wrong, it means unproven, and failures concentrate where stores are dense a

## 6. Assign per year + save raster/parquet

Two assignments per year, from the same matrix and at no extra OSRM cost:

* `a` -- rank 1 only. The winner-take-all catchment; everything in section 7 that
  allocates population reads from this.
* `a_iso` -- every store reachable within `ISO_MIN` minutes. The *local market*:
  who a cell's residents could go to, not who they are assumed to pick. Used by
  `local_market` in section 7.

`a_iso` is a top-k-then-filter, not a top-N: capping the choice set at N would put
a floor of 10,000/N under every local HHI computed from it.

`write_label_raster` needs `cells_orig` (it carries row/col) and `mapping`, so the
tif comes out at full 100 m resolution regardless of how far the solve was
coarsened. The GeoTIFF stores integer codes; the TDLinx lookup is a parquet sidecar.

In [14]:
assignments, isochrones = {}, {}
for year in YEARS:
    a = nv.assign(c2, stores, cand, dur, year=year, ranks=1)
    assignments[year] = a
    isochrones[year] = nv.assign(c2, stores, cand, dur, year=year,
                                 ranks=cand.k, max_seconds=ISO_MIN * 60)

    alive = ((stores["first_year"].to_numpy() <= year)
             & (stores["last_year"].to_numpy() >= year))
    n_orph = nv.orphans(cand, dur, alive).size
    print(f"\n=== {year} ===  {a.height:,} nodes assigned, "
          f"{int(alive.sum()):,} stores alive, {n_orph:,} orphan node(s)")
    print(nv.rank_coverage(isochrones[year]).head(8))

    for name, obj in (("catchments",  nv.catchments(a)),
                      ("assignments", a),
                      ("isochrone",   isochrones[year])):
        out_dir = OUT / name
        out_dir.mkdir(parents=True, exist_ok=True)
        obj.write_parquet(out_dir / f"{name}_{SPEC}_{year}.parquet")

    # once per year, not once per output file
    nv.write_label_raster(cells_orig, a, mapping, GHSL,
                          OUT / f"nearest_store_{SPEC}_{year}.tif")



=== 2024 ===  42,504 nodes assigned, 5,670 stores alive, 0 orphan node(s)
shape: (0, 6)
┌──────┬──────┬─────┬─────┬─────────────┬───────────┐
│ rank ┆ year ┆ n   ┆ pop ┆ share_nodes ┆ share_pop │
│ ---  ┆ ---  ┆ --- ┆ --- ┆ ---         ┆ ---       │
│ i8   ┆ i16  ┆ u32 ┆ f32 ┆ f64         ┆ f32       │
╞══════╪══════╪═════╪═════╪═════════════╪═══════════╡
└──────┴──────┴─────┴─────┴─────────────┴───────────┘


## 7. Metrics

Winner-take-all allocation: the rank-1 store takes the cell. No demand model, no
free parameter, so nothing here can be attacked on a chosen elasticity. Huff lives
in `huff.py` and is parked (section 10).

Three views of concentration, deliberately not one:

| | market | share basis |
|---|---|---|
| `hhi(market_shares(...))` | Kreis | allocated population |
| `hhi(size_shares(...))` | Kreis | sales area (VKFL) |
| `local_market(...)` | each store's own `ISO_MIN`-minute catchment | store count |

The first is comparable with official statistics; the second is the basis
competition practice actually uses; the third does not inherit administrative
borders, which is where the first two are weakest.

In [15]:
labels, lookup = M.rasterize_regions(kreis_gdf, GHSL, cells_orig,
                                     id_col=KREIS_ID, name_col=KREIS_NAME)
store_region = M.label_stores(stores, kreis_gdf, id_col=KREIS_ID)

TOTAL_POP = float(cells_orig["pop"].cast(pl.Float64).sum())

metrics = {}
for year in YEARS:
    a = assignments[year]

    # ATTRACT is time-varying, so it comes from the panel per year, never from
    # prepare_stores(keep=...). .sort("store_id") restores prepare_stores' order
    # after the join: `stores` is indexed POSITIONALLY by cand/dur, and a join is
    # not guaranteed to preserve row order. (nv now also carries store_ids on
    # Candidates and raises if a reordered frame is passed to assign/travel_times.)
    st = stores
    if ATTRACT and ATTRACT in panel.columns:
        st = (stores.join(nv.store_year_attrs(panel, year, [ATTRACT]),
                          on="store_id", how="left")
                    .sort("store_id"))
    st = st.join(store_region, on="store_id", how="left").sort("store_id")

    alloc  = M.allocate(a)                                    # winner-take-all
    ca     = M.expand_to_cells(alloc, cells_orig, mapping, labels=labels)

    # every join above is an inner join, so unmapped or unreachable population
    # leaves no trace anywhere. This is the only place it becomes visible.
    lost = 1 - float(ca["pop_alloc"].sum()) / TOTAL_POP
    print(f"{year}: {lost:.3%} of population unallocated "
          f"({lost * TOTAL_POP:,.0f} people)")
    assert lost < 0.01, ("more than 1% of population never reached a store -- check "
                         "orphans(), the mapping, and rank coverage before reading "
                         "anything below")

    shares = M.market_shares(ca, st, by="Gruppe")
    conc   = M.hhi(shares)
    totals = M.store_totals(ca, st, by="Gruppe")
    local  = M.local_market(isochrones[year], st, by="Gruppe", catchment=a)
    sz     = M.size_shares(st, by="Gruppe", size_col=ATTRACT, region_col=KREIS_ID) \
             if ATTRACT and ATTRACT in st.columns else None
    conc_sz = M.hhi(sz, region_col=KREIS_ID) if sz is not None else None

    metrics[year] = dict(stores=st, alloc=ca, shares=shares, hhi=conc, local=local,
                         totals=totals, size_shares=sz, hhi_size=conc_sz,
                         unallocated=lost)

    for name, df in (("shares", shares), ("hhi", conc), ("local_market", local),
                     ("gruppe_totals", totals), ("size_shares", sz),
                     ("hhi_size", conc_sz)):
        if df is None:
            continue
        out_dir = OUT / name
        out_dir.mkdir(parents=True, exist_ok=True)
        df.write_parquet(out_dir / f"{name}_{SPEC}_{year}.parquet")


rasterize_regions: 44 regions
label_stores: 5,670 store(s) fell outside every polygon
2024: 0.000% of population unallocated (0 people)


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

## 8. Printed summary

In [ ]:
for year in YEARS:
    m = metrics[year]
    conc, tot, loc = m["hhi"], m["totals"], m["local"]
    named = conc.join(lookup, on="region_code", how="left").filter(
        pl.col("region_code") > 0)

    print(f"\n{'='*68}\n{year}   (winner-take-all)\n{'='*68}")
    print(f"population allocated : {m['alloc']['pop_alloc'].sum():,.0f} "
          f"({1 - m['unallocated']:.2%} of raster)")
    print(f"Kreise               : {named.height}")
    print(f"HHI (Gruppe, pop)    : median {named['hhi'].median():,.0f}  "
          f"p10 {named['hhi'].quantile(0.1):,.0f}  p90 {named['hhi'].quantile(0.9):,.0f}")
    if m["hhi_size"] is not None:
        hs = m["hhi_size"]
        print(f"HHI (Gruppe, {ATTRACT})   : median {hs['hhi'].median():,.0f}  "
              f"p10 {hs['hhi'].quantile(0.1):,.0f}  p90 {hs['hhi'].quantile(0.9):,.0f}"
              f"   <- supply-side cross-check")
    print(f"CR3                  : median {named['cr3'].median():.1%}")
    n_hi = named.filter(pl.col("hhi") > 2000).height
    print(f"Kreise above HHI 2000: {n_hi} ({n_hi/named.height:.0%})")
    print(f"local HHI ({ISO_MIN} min)  : median {loc['local_hhi'].median():,.0f}, "
          f"mean {loc['n_groups'].mean():.2f} Gruppen / "
          f"{loc['n_stores'].mean():.2f} stores reachable")

    print("\nGruppe shares: allocated population vs sales area")
    cmp = tot.select("Gruppe", pl.col("share").alias("pop_share"), "pop")
    if m["size_shares"] is not None:
        vk = (m["size_shares"].group_by("Gruppe")
                .agg(pl.col("value").sum())
                .with_columns((pl.col("value") / pl.col("value").sum())
                              .alias(f"{ATTRACT}_share")))
        cmp = cmp.join(vk.select("Gruppe", f"{ATTRACT}_share"), on="Gruppe", how="left")
    print(cmp.head(10))

    print("\nMost concentrated Kreise")
    print(named.select(KREIS_NAME, "hhi", "cr1", "cr3", "n_players", "pop")
               .sort("hhi", descending=True).head(10))


## 9. Maps

In [ ]:

year = YEARS[-1]
m = metrics[year]

fig, axes = plt.subplots(1, 3, figsize=(21, 9))
for ax, (col, cmap, ttl) in zip(axes, [
        ("hhi", "OrRd",  f"HHI (Gruppe), {year}"),
        ("cr1", "PuBu",  f"CR1 -- largest chain's share, {year}"),
        ("n_players", "YlGn", f"Distinct Gruppen present, {year}")]):
    M.plot_choropleth(m["hhi"], kreis_gdf, lookup, id_col=KREIS_ID,
                      value_col=col, cmap=cmap, title=ttl, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# National (or widest-possible) catchment map, coloured by Gruppe
M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                  boundaries=kreis_gdf, title=f"Nearest-store catchments, {year}", save_path=OUT / "figures" / f"catchments_{SPEC}_{year}.png")
plt.show()

In [ ]:
# Zoom to individual Kreise. Iterate on AGS5: it is the only key -- Kreis NAMES
# repeat within Bayern (München is both 09162 and 09184), so filtering figures on
# the name can silently select two Kreise and draw their combined bounding box.
for row in kreis_gdf[["AGS5", "KREIS_LABEL"]].itertuples(index=False):
    ax = M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                           region=kreis_gdf, region_col="AGS5", region_name=row.AGS5,
                           title=f"{row.KREIS_LABEL}, {year}",
                           save_path=FIGS / f"catchments_{SPEC}_{row.AGS5}_{year}.png")
    plt.show()
    plt.close(ax.figure)          # ~96 Kreise: without this they all stay open


## 10. Robustness

The headline is winner-take-all, so there is no elasticity to vary — the
sensitivity that remains is to the *market definition*, not to a demand parameter:
how wide the local isochrone is, and whether shares are read off population or
sales area.

Probabilistic (Huff) allocation has moved to `huff.py` and is parked. Two things
have to be settled before a number from it is reportable: `beta` is assumed rather
than estimated, and the choice set is truncated by `k`, which biases every
retained store's share upward. `huff.truncation_report()` quantifies the second.
When it is ready it returns the same `(cell_id, store_id, share)` frame, so it
drops into the cell above with one line changed:

```python
import huff
alloc = huff.allocate(isochrones[year], st, attract_col=ATTRACT, beta=2.0)
```

In [ ]:
st = metrics[year]["stores"]
rows = []
for mins in (10, 15, 20, 25, 30):
    aI = nv.assign(c2, stores, cand, dur, year=year, ranks=cand.k,
                   max_seconds=mins * 60)
    lm = M.local_market(aI, st, by="Gruppe", catchment=assignments[year])
    lv = M.local_market(aI, st, by="Gruppe", size_col=ATTRACT,
                        catchment=assignments[year]) \
         if ATTRACT and ATTRACT in st.columns else None
    rows.append({
        "isochrone_min": mins,
        "local_hhi_median": lm["local_hhi"].median(),
        f"local_hhi_{ATTRACT}_wtd": lv["local_hhi"].median() if lv is not None else None,
        "mean_gruppen": lm["n_groups"].mean(),
        "mean_stores": lm["n_stores"].mean(),
        "monopoly_share": lm.filter(pl.col("local_hhi") >= 9_999).height / lm.height,
    })

print(pl.DataFrame(rows))


In [ ]:
st.unique(pl.col("store_id"))
st.filter(pl.col("VKFL").is_null()).describe()